# 🚀 Python İleri Seviye Konular

Bu bölümde Python'un güçlü ama sık atlanan kavramlarını ele alacağız.
Kuru tanımlar yerine **neden var, ne zaman lazım, nasıl kullanılır** sorularına odaklanacağız.

---

## 🔹 1. Decorators (Dekoratörler)

### Sezgisel Başlangıç

Şöyle düşün: Bir kafe açtın ve her sipariş geldiğinde hem kasaya kayıt etmek hem de
siparişi hazırlamak istiyorsun. Her fonksiyonun içine `print("kayıt edildi")` yazmak
yerine bunu **merkezi** bir yere toplamak çok daha temiz olur. İşte decorator tam bu iş için var.

Python'da fonksiyonlar **birinci sınıf vatandaşlardır** — yani bir fonksiyonu
başka bir fonksiyona parametre olarak verebilirsin. Decorator'lar bu özelliği kullanır.
```python
def kayit_et(func):
    def wrapper(*args, **kwargs):
        print(f"📋 '{func.__name__}' çağrıldı")
        sonuc = func(*args, **kwargs)
        print(f"✅ '{func.__name__}' tamamlandı")
        return sonuc
    return wrapper

@kayit_et
def siparis_al(urun):
    print(f"☕ {urun} hazırlanıyor...")

siparis_al("Latte")
```
```
📋 'siparis_al' çağrıldı
☕ Latte hazırlanıyor...
✅ 'siparis_al' tamamlandı
```

### Perde Arkasında Ne Oluyor?

`@kayit_et` yazmak aslında şu satırla **birebir aynıdır:**
```python
siparis_al = kayit_et(siparis_al)
```

`@` sadece bunu daha okunabilir hale getiren bir sözdizimi (syntactic sugar).

### Neden `*args, **kwargs`?

`wrapper(*args, **kwargs)` yazmazsak dekoratör sadece
parametresiz fonksiyonlarla çalışır. `*args, **kwargs` ile **her türlü** fonksiyonu sarabilir hale geliriz.

### 🎯 Gerçek Hayatta Nerede Karşına Çıkar?
```python
# Zamanlama — bir fonksiyonun ne kadar sürdüğünü ölç
import time

def zamanla(func):
    def wrapper(*args, **kwargs):
        baslangic = time.time()
        sonuc = func(*args, **kwargs)
        sure = time.time() - baslangic
        print(f"⏱️ {func.__name__} → {sure:.4f} saniye")
        return sonuc
    return wrapper

@zamanla
def agir_hesaplama(n):
    return sum(range(n))

agir_hesaplama(10_000_000)
# ⏱️ agir_hesaplama → 0.2341 saniye
```

Decorator'ı bir kez yaz, **istediğin kadar fonksiyona** uygula.

---

## 🔹 2. Encapsulation (Kapsülleme)

### Sorun: Kontrolsüz Erişim
```python
class BankaHesabi:
    def __init__(self, bakiye):
        self.bakiye = bakiye  # tamamen açık!

hesap = BankaHesabi(1000)
hesap.bakiye = -999999  # 😱 hiçbir engel yok
```

Gerçek hayatta böyle bir banka olmaz. Bakiyeye doğrudan müdahale edilemez,
sadece **para yatır / para çek** gibi kontrollü işlemler yapılabilir.
Python'da bu korumayı `__` (çift alt çizgi) ile sağlarız:
```python
class BankaHesabi:
    def __init__(self, bakiye):
        self.__bakiye = bakiye  # artık dışarıdan erişilemez

hesap = BankaHesabi(1000)
print(hesap.__bakiye)  # ❌ AttributeError
```

> `__bakiye` dışarıdan **görünmez** hale gelir.
> Python bunu `_BankaHesabi__bakiye` adıyla gizler — buna **name mangling** denir.

---

## 🔹 3. Property — Modern Getter

Veriyi gizledik, güzel. Ama artık dışarıdan hiç okunamıyor.
Bunu çözmek için **kontrollü bir okuma kapısı** açmamız gerekiyor.

### Eski Yol — Getter Metodu
```python
def get_bakiye(self):
    return self.__bakiye

hesap.get_bakiye()  # fonksiyon gibi çağrılıyor
```

Bu çalışır ama çirkin. Java tarzı bir yaklaşım.

### Modern Yol — `@property`
```python
class BankaHesabi:
    def __init__(self, bakiye):
        self.__bakiye = bakiye

    @property
    def bakiye(self):
        return self.__bakiye
```
```python
hesap = BankaHesabi(500)
print(hesap.bakiye)   # ✅ 500 — parantez yok, attribute gibi davranıyor
hesap.bakiye = 999    # ❌ AttributeError: has no setter
```

> `@property` sayesinde method, **attribute gibi** çağrılır.
> Setter tanımlanmadığı sürece değer **salt okunur (read-only)** olur.

---

## 🔹 4. Setter

Salt okunur bakiye işimize yaramaz, para hareketi gerekiyor.
Ama doğrudan atamaya da izin vermek istemiyoruz.
**Setter** tam bu noktada devreye girer:
```python
class BankaHesabi:
    def __init__(self, bakiye):
        self.__bakiye = bakiye

    @property
    def bakiye(self):
        return self.__bakiye

    @bakiye.setter
    def bakiye(self, yeni_deger):
        self.__bakiye = yeni_deger
```
```python
hesap = BankaHesabi(500)
hesap.bakiye = 1500     # ✅ artık değiştirilebilir
print(hesap.bakiye)     # 1500
```

> ⚠️ Setter dekoratörünün adı `@property` değil, **`@bakiye.setter`** olmalı.
> Hangi property için setter yazıyorsan onun adını kullanırsın.

---

## 🔹 5. Data Validation — Setter'ın Asıl Gücü

Setter yazdık ama hâlâ şunu yapabiliyoruz:
```python
hesap.bakiye = -50000  # mantıksız ama hata vermez
hesap.bakiye = "beş lira"  # tamamen saçma ama kabul edilir
```

Setter'ın gerçek değeri burada ortaya çıkar — **geçersiz veriyi daha girerken engellemek:**
```python
class BankaHesabi:
    def __init__(self, sahip, bakiye):
        self.__sahip = sahip
        self.__bakiye = bakiye

    @property
    def bakiye(self):
        return self.__bakiye

    @bakiye.setter
    def bakiye(self, miktar):
        if not isinstance(miktar, (int, float)):
            raise TypeError("Bakiye sayısal bir değer olmalı")
        if miktar < 0:
            raise ValueError("Bakiye negatif olamaz")
        self.__bakiye = miktar

    @property
    def sahip(self):
        return self.__sahip

    @sahip.setter
    def sahip(self, isim):
        if not isinstance(isim, str) or len(isim) < 2:
            raise ValueError("Geçerli bir isim giriniz")
        self.__sahip = isim
```
```python
hesap = BankaHesabi("Ahmet", 1000)
hesap.bakiye = -500       # ❌ ValueError: Bakiye negatif olamaz
hesap.bakiye = "bin lira" # ❌ TypeError: Bakiye sayısal bir değer olmalı
hesap.sahip = "A"         # ❌ ValueError: Geçerli bir isim giriniz
```

### 💡 Kritik İncelik — `__init__` İçinde Setter Kullan
```python
# ❌ Yanlış — validation bypass edilir
def __init__(self, bakiye):
    self.__bakiye = bakiye  # setter'ı atladık, kural yok

# ✅ Doğru — validation baştan çalışır
def __init__(self, bakiye):
    self.bakiye = bakiye  # setter üzerinden geçer, kurallar geçerli
```

---

## 🔹 6. Static Methods

Kafe örneğimize dönelim. Müşterinin ödeyeceği KDV'yi hesaplayalım.
Bu hesaplama için **belirli bir siparişe ya da kasaya ihtiyaç yok** — sadece bir sayı gelecek, sonuç döndürülecek.
```python
class Kafe:

    def __init__(self, isim):
        self.isim = isim

    @staticmethod
    def kdv_hesapla(tutar, oran=0.18):
        return round(tutar * (1 + oran), 2)

    @staticmethod
    def gecerli_odeme_mi(miktar):
        return isinstance(miktar, (int, float)) and miktar > 0
```
```python
# Instance oluşturmadan doğrudan kullanılır
print(Kafe.kdv_hesapla(100))        # 118.0
print(Kafe.gecerli_odeme_mi(-50))   # False
```

> Static method'ların `self` ya da `cls` parametresi **yoktur.**
> Class içinde mantıksal olarak gruplandırılmış, ama class/instance'ın **durumundan bağımsız** fonksiyonlardır.

---

## 🔹 7. Class Methods ve Alternative Constructor

### Class'ın Kendi Durumunu Yönetmek
```python
class Kafe:
    toplam_siparis = 0  # tüm instance'lar arasında paylaşılan class değişkeni

    def __init__(self, urun):
        self.urun = urun
        Kafe.toplam_siparis += 1

    @classmethod
    def kac_siparis(cls):
        return cls.toplam_siparis
```
```python
s1 = Kafe("Latte")
s2 = Kafe("Espresso")
s3 = Kafe("Filtre")

print(Kafe.kac_siparis())  # 3 ✅
```

### Alternative Constructor — Farklı Kaynaklardan Obje Üretmek

Kullanıcı bazen `dict`, bazen JSON string, bazen CSV satırı gönderebilir.
Her seferinde `__init__`'i şişirmek yerine **ayrı constructor'lar** tanımlayabiliriz:
```python
class Kullanici:

    def __init__(self, isim, email):
        self.isim = isim
        self.email = email

    @classmethod
    def sozlukten(cls, veri: dict):
        return cls(veri["isim"], veri["email"])

    @classmethod
    def metinden(cls, metin: str):
        # "Ahmet:ahmet@mail.com" formatı
        isim, email = metin.split(":")
        return cls(isim, email)

    def __repr__(self):
        return f"Kullanici({self.isim}, {self.email})"
```
```python
k1 = Kullanici("Zeynep", "zeynep@mail.com")
k2 = Kullanici.sozlukten({"isim": "Mert", "email": "mert@mail.com"})
k3 = Kullanici.metinden("Elif:elif@mail.com")

print(k1)  # Kullanici(Zeynep, zeynep@mail.com)
print(k2)  # Kullanici(Mert, mert@mail.com)
print(k3)  # Kullanici(Elif, elif@mail.com)
```

> `cls(...)` kullanmak, `Kullanici(...)` yazmaktan daha doğrudur.
> Class kalıtımla genişletildiğinde `cls` otomatik olarak **doğru alt class'ı** işaret eder.

---

## 📊 Genel Karşılaştırma

| Kavram | Anahtar Kelime | `self` | `cls` | Ne İşe Yarar |
|---|---|:---:|:---:|---|
| Normal Method | — | ✅ | ❌ | Instance verisiyle çalışır |
| Property | `@property` | ✅ | ❌ | Attribute gibi görünen getter |
| Setter | `@x.setter` | ✅ | ❌ | Kontrollü & validasyonlu atama |
| Static Method | `@staticmethod` | ❌ | ❌ | Bağımsız yardımcı fonksiyon |
| Class Method | `@classmethod` | ❌ | ✅ | Class state & alt. constructor |

---

## 🎯 Akılda Kalması Gereken 5 Kural

1. **Decorator** → Fonksiyona dokunmadan ek sorumluluk yükle
2. **`__` prefix** → Veriyi gizle, doğrudan erişimi kapat
3. **`@property` + `@x.setter`** → Gizli veriye temiz ve kontrollü bir arayüz sun
4. **Setter içinde validation yap, `__init__`'te setter'ı kullan** → Hatalı veri **hiç giremez**
5. **`@staticmethod` vs `@classmethod`** → Class'a ihtiyaç var mı? → `classmethod`; yok mu? → `staticmethod`

### Decorators

In [1]:
# https://medium.com/@syndrome/python-decorators-fb7d906890dc

In [2]:
def my_decorator(func):

    def wrapper():
        print('wrapper function executed')
        func()
        print('wrapper function executed')

    return wrapper

In [3]:
@my_decorator
def hello_world():
    print("Hello, World!")

In [4]:
hello_world()

wrapper function executed
Hello, World!
wrapper function executed


### Getter / Setter

In [5]:
# Property Decorators
# data validation, private / public (encapsulation)

class Person():

    def __init__(self, name, age):
        self.__name = name # __name -> private, sadece tanımlı class içerisinden erişilebilir | name -> public, her yerden erişilebilir.
        self.__age = age

    def get_name(self):
        return self.__name

In [6]:
my_name = Person("Ensar", 22)
my_name.get_name()

'Ensar'

In [7]:
class Person():

    def __init__(self, name, age):
        self.__name = name 
        self.__age = age

    # getter
    @property
    def name(self):
        return self.__name

In [8]:
my_name = Person("Arda", 21)
my_name.name

'Arda'

In [9]:
# my_name.name = "Kenan" 
# name değiştirmek istedğimizi zaman alınan hata: AttributeError: property 'name' of 'Person' object has no setter

In [10]:
class Person():

    def __init__(self, name, age):
        self.__name = name 
        self.__age = age

    # getter
    @property
    def name(self):
        return self.__name

    @name.setter
    def name(self, value):
        self.__name = value

In [11]:
my_name = Person("Ensar", 35)
print(my_name.name)
my_name.name = "Süleyman"
print(my_name.name)

Ensar
Süleyman


### Data Validation

In [12]:
class Person():

    def __init__(self, name, age):
        self.__name = name 
        self.__age = age

    # getter
    @property
    def name(self):
        return self.__name

    @name.setter
    def name(self, value):
        if not isinstance(value, str):
            raise ValueError("Name must be a string")
        if len(value) < 2:
            raise ValueError("Name should be longer")
        self.__name = value

In [13]:
my_name = Person("Kamil", 11)
my_name.name = 70

ValueError: Name must be a string

In [14]:
my_name.name = "A"

ValueError: Name should be longer

In [15]:
class Person():

    def __init__(self, name, age):
        self.__name = name 
        self.__age = age

    # getter
    @property
    def name(self):
        return self.__name

    @name.setter
    def name(self, value):
        if not isinstance(value, str):
            raise ValueError("Name must be a string")
        if len(value) < 2:
            raise ValueError("Name should be longer")
        self.__name = value

    @property
    def age(self):
        return self.__age

    @age.setter
    def age(self, value):
        if not isinstance(value, int):
            raise ValueError("Age must be a integer")
        if value < 0:
            raise ValueError("Age should be +")
        self.__age = value

In [16]:
my_name = Person("Ensar", 22)
print(my_name.name)
print(my_name.age)

Ensar
22


In [17]:
my_name.age = 15
my_name.age

15

### Static Method

In [18]:
class MathOperations:

    @staticmethod
    def add(x, y):
        return x + y

    @staticmethod
    def divide(x, y):
        return x / y

In [19]:
print("Toplama:", MathOperations.add(2, 3))
print("Bölme:", MathOperations.divide(10, 2))

Toplama: 5
Bölme: 5.0


### Class Method

In [20]:
# alternative constructor

class Pizza:

    total_pizzas = 0

    def __init__(self, ingredients):
        self.ingredients = ingredients
        Pizza.total_pizzas += 1

    @classmethod
    def margarita(cls):
        return cls(['peynir', 'domates', 'fesleğen'])

    @classmethod
    def pepperoni(cls):
        return cls(['peynir', 'domates', 'sucuk'])

    @classmethod
    def get_total_pizzas(cls):
        return cls.total_pizzas

In [21]:
pizza1 = Pizza.margarita()
print(pizza1.ingredients)
pizza2 = Pizza.pepperoni()
print(pizza2.ingredients)
print(pizza1.get_total_pizzas())

['peynir', 'domates', 'fesleğen']
['peynir', 'domates', 'sucuk']
2
